# Targeted Advertising Mean-Field Control Benchmark

Reference: Meunier, Pham & Reisinger, discrete-space benchmarks, Section "Targeted advertising with social influence" (`files/reference/discrete_benchmarks.tex`), introduced by Motte & Pham. A company repeatedly advertises to increase its customer proportion while controlling expenditure.

**Model.** State space $\mathcal X=\{0,1\}$ (not-customer / customer), action space $\mathcal A=\{0,1\}$ (no ad / ad). Unlike two-state or distribution planning, the mean-field interaction lives partly in the **transition kernel**, not only the reward:
$$P(1\mid x,a,\mu) = \min\{\mu(1) + \kappa_\mathrm{ad}a,\, 1\}, \qquad r(x,a,\mu) = x - c_\mathrm{ad}a, \qquad g(x,\mu)=0,$$
independent of $x$: an advertisement raises *everyone's* conversion/retention probability by $\kappa_\mathrm{ad}$, and the current customer proportion $\mu(1)$ itself drives a positive social-influence effect. The reward doesn't reference $\mu$ directly, but (unlike cybersecurity, where the reward also doesn't depend on the action) it does depend on the action $a$, and the *policy* is itself $\mu$-dependent, so the policy-averaged reward $R_t^\theta(i,m)=\mathbb E_{a\sim\pi_t^\theta(\cdot\mid i,m)}[i-c_\mathrm{ad}a]$ still genuinely depends on $m$. Rewards are discounted by $\gamma=0.5$ (truncating the source infinite-horizon problem to $T=5$).

**Policy.** Because only the aggregate advertising rate enters the population recursion, the optimal policy class is **individual-state-independent**: $\pi_\theta(1\mid t,x,\mu)=q_\theta(t/T,\mu(1))$ for every $x$. $q_\theta$ is a small 2-hidden-layer MLP (width 32, $\tanh$, sigmoid output, ~1.2k parameters) taking normalized time and the customer proportion.

**Unlike cybersecurity/distribution planning, this benchmark has a known closed-form structural benchmark**: the source model's stationary infinite-horizon optimal advertising probability $\hat q(p)$ (`Advertising.reference_policy`), a three-regime piecewise function of the customer proportion $p$. At this benchmark's parameters ($\kappa_\mathrm{ad}=0.2$, $c_\mathrm{ad}=0.15$, $\gamma=0.5$), it works out to
$$\hat q(p) = \begin{cases}1, & p<0.6 \\ (0.8-p)/0.2, & 0.6\le p<0.7 \\ 1, & 0.7\le p<0.85 \\ 0, & p\ge 0.85\end{cases}$$
a non-monotone rule this notebook compares the learned policy against directly (though $\hat q$ is the *infinite-horizon*, *stationary* optimum, so it isn't literally the exact target of this finite-horizon, time-dependent training problem).

**This notebook shows `main`-tier results automatically whenever they're available.** `configs/advertising.py`'s `MAIN` runs 5 seeds at $n_\mathrm{train}=10{,}000$ over the single $(T=5,\text{equal\_budget},\text{particle})$ group — no horizon or budget sweep, and unlike `mid` (which stays at `equal_parameters`/`exact`) it matches every algorithm to mfreinforce's per-step transition budget, so simplex trains at $B=6190$ and reinforce at $B=6200$; run `scripts/train_all.sh <workers> --env advertising --alg <alg> --config main` (once per algorithm) to populate it.

In [ ]:
import sys
import time
from pathlib import Path

_notebook_start = time.perf_counter()

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
for path in (SRC, ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torch
import pandas as pd
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)
torch.set_default_device("cuda" if torch.cuda.is_available() else "cpu")

from configs.advertising import MAIN, MID
from mfc.environments.advertising import AD, CUSTOMER, NO_AD, NOT_CUSTOMER, Advertising, AdvertisingConfig
from mfc.algorithms import simplex
from mfc.plotting import diagnostics as viz
from mfc.plotting.style import apply_style, color_for, new_figure, set_style, style_legend
from scripts.train import run_all
from scripts.test import (
    exact_gradient,
    exact_sensitivity_flow,
    generalization_eval,
    gradient_diagnostics,
    group_by,
    load_runs,
    logit_perturbation_coverage,
    objective_gap,
    oracle_gradient_estimate,
    perturbation_coverage,
    reference_policy_fn,
    rollout,
    sensitivity_estimation_error,
    state_distribution,
    state_marginal_stability,
)

set_style()

DEFAULT_T = MID.horizons[0]  # 5: the only horizon this benchmark uses, at any tier

## Configuration and budget

Auto-detects whether `runs/advertising/main/` has any saved runs and uses `main` if so (5 seeds, $n_\mathrm{train}=10{,}000$), otherwise `mid` (1 seed, $n_\mathrm{train}=5{,}000$). There is only one $(T,\text{budget\_mode},\text{flow})$ group at either tier, so there is no horizon-scaling or budget/flow-comparison section here.

In [ ]:
main_dir = ROOT / "runs" / "advertising" / "main"
tier = "main" if list(main_dir.glob("*_seed*.pt")) else "mid"
cfg = MAIN if tier == "main" else MID

print(f"tier: {tier}")
print(f"algorithms:    {cfg.algorithms}")
print(f"lambdas:       {cfg.lambdas}  (simplex perturbation scale)")
print(f"epsilon:       {cfg.epsilon}  (logit perturbation scale, fixed -- not swept like lambda)")
print(f"T={cfg.horizons[0]}")
print(f"seeds:         {cfg.seeds}")
print(f"B={cfg.B}, n_aux={cfg.n_aux}, sigma={cfg.sigma}, lr={cfg.lr}, n_train={cfg.n_train}")
print(f"mu0 ~ (1-p0,p0), p0~U([0.05,0.95]) during training; validation mu0={cfg.mu0_val}")

env_preview = Advertising()
print(f"kappa_ad={env_preview.config.kappa_ad}, c_ad={env_preview.config.c_ad}, gamma={env_preview.config.gamma}")
if tier == "mid":
    print("\n(no runs/advertising/main/ data yet, showing mid-tier results; run scripts/train_all.sh "
          "<workers> --env advertising --alg <alg> --config main, once per algorithm, for real seed-to-seed statistics)")

## Train (or load cached results)

At `mid`, trains first if nothing is cached yet. At `main`, only loads what's already there. Adapts `env`'s dtype to match whatever's loaded.

In [ ]:
env = Advertising()
runs_dir = ROOT / "runs" / "advertising" / tier

runs = []
for alg in cfg.algorithms:
    if tier == "mid" and not list(runs_dir.glob(f"{alg}_*_seed*.pt")):
        run_all("advertising", alg, "mid")
    runs += load_runs("advertising", alg, tier)

if runs and runs[0]["theta_final"].dtype != env.dtype:
    run_dtype = runs[0]["theta_final"].dtype
    print(f"note: loaded runs are {run_dtype}, switching env and the notebook's default dtype to match (was {env.dtype})")
    torch.set_default_dtype(run_dtype)
    env = Advertising(dtype=run_dtype)

total_train_seconds = sum(r["elapsed_seconds"] for r in runs)
print(f"{len(runs)} runs loaded ({tier}); total training compute time: {total_train_seconds:.1f}s ({total_train_seconds / 60:.1f} min)")

## Ground truth: the closed-form reference policy

The infinite-horizon optimal advertising rate $\hat q(p)$ (reference "Infinite-horizon reference policy"), and its induced trajectory from the validation initial law: the reference line every section below compares against.

In [ ]:
ps = torch.linspace(0.0, 1.0, 201, dtype=env.dtype, device=env.device)
q_hat = env.reference_policy(ps)

fig, ax = new_figure()
ax.plot(ps.cpu(), q_hat.cpu(), color=color_for(0), linewidth=2, label="q_hat(p) (closed-form optimal)")
apply_style(ax, xlabel="customer proportion p", ylabel="advertising probability q_hat(p)", title="Reference advertising policy")
style_legend(ax)

mu0_val = torch.tensor(cfg.mu0_val, dtype=env.dtype, device=env.device)
reference_traj = state_distribution(env, reference_policy_fn(env), env.init_theta(), mu0_val, DEFAULT_T)
print("customer proportion under q_hat, starting from mu0_val:", reference_traj[:, CUSTOMER].tolist())

## Default group

`by_lambda` picks seed 0 as a representative $\theta$ per $\lambda$; `by_lambda_all_seeds` keeps every seed for the aggregate diagnostics, which cover every $\lambda$ throughout this notebook, not just one.

In [ ]:
simplex_runs = [r for r in runs if r["alg"] == "simplex"]
mfreinforce_runs = [r for r in runs if r["alg"] == "mfreinforce"]
reinforce_runs = [r for r in runs if r["alg"] == "reinforce"]

by_lambda_all_seeds = {lam: grp for (lam,), grp in group_by(simplex_runs, "lam").items()}
by_lambda = {lam: next((r for r in grp if r["seed"] == 0), grp[0]) for lam, grp in by_lambda_all_seeds.items()}
theta_02 = by_lambda[0.2]["theta_final"]

print(f"{len(simplex_runs)} simplex runs across {len(by_lambda)} lambda values, "
      f"{len(mfreinforce_runs)} mfreinforce run(s), {len(reinforce_runs)} reinforce run(s)")

## Evolution of the validation reward

The exact validation objective $J_T(\theta_m;\mu_0^\mathrm{val})$ every 10 training iterations, one line per simplex $\lambda$ plus reinforce and mfreinforce. At `main`, each line is the mean $\pm$ 1 std across 5 seeds.

In [ ]:
fig, ax = viz.plot_validation_curve(runs)
ax.set_title(f"Validation objective by training iteration ({tier} tier)")

## Learned policy vs. the closed-form reference, across time

$q_\theta(t/T, p)$ for $\lambda=0.2$'s learned policy, at a few decision times $t$, against the stationary $\hat q(p)$. Since $\hat q$ is the *infinite-horizon* optimum, the learned finite-horizon policy is expected to diverge from it near the terminal boundary $t=T-1$ (no continuation value left to protect).

In [ ]:
fig, ax = new_figure()
ax.plot(ps.cpu(), q_hat.cpu(), color="black", linestyle="--", linewidth=2, label="q_hat(p) (reference)")
for i, t in enumerate([0, DEFAULT_T // 2, DEFAULT_T - 1]):
    q_learned = torch.stack([env.policy_probs(theta_02, t, torch.tensor(NOT_CUSTOMER, device=env.device), torch.stack([1 - p, p]))[AD] for p in ps])
    ax.plot(ps.cpu(), q_learned.detach().cpu(), color=color_for(i + 1), linewidth=2, label=f"q_theta(t={t}, p)")
apply_style(ax, xlabel="customer proportion p", ylabel="advertising probability", title="Advertising probability vs customer share (lambda=0.2)")
style_legend(ax)

## Customer-proportion trajectory: learned vs reference

The exact population flow (customer proportion $\mu_t(1)$) from $\mu_0^\mathrm{val}$, under the learned policy against the closed-form reference policy: the graph shows $\lambda=0.2$ specifically (learned, solid; reference, dashed); the table gives every $\lambda$.

In [ ]:
learned_flow = state_distribution(env, env.policy_probs, theta_02, mu0_val, DEFAULT_T)
fig, ax = viz.plot_state_distribution(learned_flow, optimal_flow=reference_traj, state_labels=["not customer", "customer"])
ax.set_title("Customer share over the validation horizon")

rows = []
for lam in sorted(by_lambda):
    flow = state_distribution(env, env.policy_probs, by_lambda[lam]["theta_final"], mu0_val, DEFAULT_T)
    rows.append({"theta": f"λ={lam}", **{f"mu_{t}(customer)": flow[t, CUSTOMER].item() for t in range(flow.shape[0])}})
rows.append({"theta": "reference (q_hat)", **{f"mu_{t}(customer)": reference_traj[t, CUSTOMER].item() for t in range(reference_traj.shape[0])}})
pd.DataFrame(rows).set_index("theta")

## $J^\lambda$ vs $J$, and gradient bias/variance

The simplex plug-in gradient estimator (`mfc.algorithms.simplex.gradient_estimate`) is

$$\hat g_{B,n,\lambda,\eta}(\theta) = \frac1B\sum_{b=1}^B\sum_{t=0}^{T}\Big[\mathbb 1_{\{t<T\}}L_t^{(b)} + Q_t^{(b)}\Big]\,G_t^{(b)},$$

with $L_t=\nabla_\theta\log\pi_t^\theta(a_t\mid x_t,M_t)$ the direct policy score, $Q_t=-\frac{1-\lambda}{\lambda}H(q_t)^\top\hat D_t$ the population-sensitivity correction, and $G_t$ the return-to-go. Both diagnostics below are at every $\lambda$'s own learned $\hat\theta_\lambda$ (seed 0), summarized as $\|\text{bias}\|$/$\|\text{std}\|$ over the ~1.2k-dimensional MLP parameter vector; only simplex has this plug-in estimator, so this table stays simplex-only.

In [ ]:
gaps, grad_diag = {}, {}
for lam, r in by_lambda.items():
    theta = r["theta_final"]
    gaps[lam] = objective_gap(env, env.policy_probs, theta, mu0_val, DEFAULT_T, lam=lam, sigma=cfg.sigma, n_samples=5000)
    grad_diag[lam] = gradient_diagnostics(
        env, env.policy_probs, theta, mu0_val, DEFAULT_T,
        lam=lam, n_aux=cfg.n_aux, B=cfg.B, sigma=cfg.sigma, reps=30,
    )

rows = []
for lam in sorted(gaps):
    g, d = gaps[lam], grad_diag[lam]
    rows.append({
        "λ": lam,
        "J(theta_hat)": g["J"].item(), "J^lambda(theta_hat) (MC)": g["J_lambda_mean"].item(), "|gap|": abs(g["gap"].item()),
        "||bias||": d["bias"].norm().item(), "||std||": d["std"].norm().item(),
    })
pd.DataFrame(rows).set_index("λ")

## Perturbation coverage: simplex $d_{TV}(M^\lambda,\mu)\le\lambda$ and mfreinforce $\mathbb E[d_{TV}]\le\varepsilon/2$

Checked (as elsewhere) by direct sampling at a few representative population laws. Simplex's bound holds *almost surely* (every draw); mfreinforce's logit perturbation only holds *in expectation* (`files/Discrete RL - Meunier, Pham, Reisinger.md`, Lemma 2.2).

In [ ]:
mu_labels = ["mu0_val", "mostly non-customers", "uniform"]
mu_samples = torch.stack([mu0_val, torch.tensor([0.9, 0.1], dtype=env.dtype, device=env.device), torch.tensor([0.5, 0.5], dtype=env.dtype, device=env.device)])

coverage = perturbation_coverage(mu_samples, lam=0.2, sigma=cfg.sigma, n_samples=5000)
for r in coverage:
    assert r["within_bound"], "the perturbation theorem's bound should never be violated"
print("simplex, lambda=0.2:")
display(pd.DataFrame([{"mu": lbl, "mean_dTV": r["mean_dTV"].item(), "max_dTV": r["max_dTV"].item(), "bound (lambda)": 0.2, "within_bound": r["within_bound"]}
                       for lbl, r in zip(mu_labels, coverage)]).set_index("mu"))
fig, ax = viz.plot_perturbation_coverage(coverage, 0.2, mu_labels=mu_labels)
ax.set_title("Simplex perturbation coverage against the TV bound")

logit_coverage = logit_perturbation_coverage(mu_samples, epsilon=cfg.epsilon, n_samples=5000)
for r in logit_coverage:
    assert r["within_bound"], "Lemma 2.2's expected-value bound should hold at this sample size"
print("\nmfreinforce, epsilon={}:".format(cfg.epsilon))
display(pd.DataFrame([{"mu": lbl, "mean_dTV": r["mean_dTV"].item(), "max_dTV": r["max_dTV"].item(), "bound (epsilon/2)": cfg.epsilon / 2, "within_bound (mean)": r["within_bound"]}
                       for lbl, r in zip(mu_labels, logit_coverage)]).set_index("mu"))

## Sample trajectories: learned vs reference policy

One sampled state trajectory under the learned policy ($\lambda=0.2$, seed 0) and one under the closed-form reference policy, both from $\mu_0^\mathrm{val}$.

In [ ]:
learned_traj = rollout(env, env.policy_probs, theta_02, mu0_val, T=DEFAULT_T, generator=torch.Generator(device=env.device).manual_seed(0))
reference_traj_sample = rollout(env, reference_policy_fn(env), env.init_theta(), mu0_val, T=DEFAULT_T, generator=torch.Generator(device=env.device).manual_seed(1))
fig, ax = viz.plot_trajectories(learned_traj, reference_traj_sample)
ax.set_yticks([NOT_CUSTOMER, CUSTOMER])
ax.set_yticklabels(["not customer", "customer"])
ax.set_title("Sample customer-state path: learned vs reference")

## Generalization without retraining

Evaluating every $\lambda$'s learned $\hat\theta_\lambda$ (seed 0) exactly (no retraining) under different initial laws, a longer horizon, and model misspecification (shifted advertising efficiency/cost).

In [ ]:
scenarios = [
    {"name": "baseline (mu0_val)"},
    {"name": "mu0=mostly non-customers", "mu0": torch.tensor([0.9, 0.1], dtype=env.dtype, device=env.device)},
    {"name": "mu0=mostly customers", "mu0": torch.tensor([0.1, 0.9], dtype=env.dtype, device=env.device)},
    {"name": "T=10", "T": 10},
    {"name": "2x advertising efficiency", "env": Advertising(AdvertisingConfig(kappa_ad=0.4), dtype=env.dtype, device=env.device)},
    {"name": "2x advertising cost", "env": Advertising(AdvertisingConfig(c_ad=0.3), dtype=env.dtype, device=env.device)},
]

rows = {sc["name"]: {"scenario": sc["name"]} for sc in scenarios}
for lam in sorted(by_lambda):
    gen_results = generalization_eval(env, env.policy_probs, by_lambda[lam]["theta_final"], mu0_val, DEFAULT_T, scenarios)
    for res in gen_results:
        rows[res["name"]][f"λ={lam}"] = res["J"].item()
pd.DataFrame(list(rows.values())).set_index("scenario")

## Comparing the three algorithms

Final validation objective for each algorithm, and each one's distance from the closed-form reference trajectory. Mean $\pm$ std across seeds at `main`; a single value at `mid` (which can't separate genuine differences from noise).

In [ ]:
def flow_err(theta):
    flow = state_distribution(env, env.policy_probs, theta, mu0_val, DEFAULT_T)
    return (flow[:, CUSTOMER] - reference_traj[:, CUSTOMER]).abs().mean().item()

rows = []
for lam, group in sorted(by_lambda_all_seeds.items()):
    vals = torch.tensor([r["validation_J"][-1].item() for r in group])
    errs = torch.tensor([flow_err(r["theta_final"]) for r in group])
    rows.append({"algorithm": f"simplex λ={lam}", "final validation J": vals.mean().item(), "std": vals.std().item() if len(vals) > 1 else 0.0,
                 "mean |p_t - p_t^ref|": errs.mean().item()})
for alg, group in (("mfreinforce", mfreinforce_runs), ("reinforce", reinforce_runs)):
    vals = torch.tensor([r["validation_J"][-1].item() for r in group])
    errs = torch.tensor([flow_err(r["theta_final"]) for r in group])
    rows.append({"algorithm": alg, "final validation J": vals.mean().item(), "std": vals.std().item() if len(vals) > 1 else 0.0,
                 "mean |p_t - p_t^ref|": errs.mean().item()})
pd.DataFrame(rows).set_index("algorithm")

## Additional statistical validation (discrete-state theory)

The sections above check the estimators against each other and against training outcomes. The sections below instead check the theory's own asymptotic claims from `files/reference/discrete_state_space(2).tex` directly, at a single fixed $\theta=\hat\theta_{0.2}$ (`theta_02`) so $\lambda$ is the only thing varying. As with cybersecurity, advertising's mean-field coupling lives partly in the **transition kernel** ($P(1\mid x,a,\mu)$ depends on $\mu(1)$), and unlike cybersecurity, the *reward* also has real $\mu$-dependence once averaged over the ($\mu$-dependent) policy, since $r$ depends on the action $a$: $R_t^\theta(i,m)=\mathbb E_{a\sim\pi_t^\theta(\cdot|i,m)}[i-c_\mathrm{ad}a]$ is not constant in $m$ here.

This MLP-policy environment is more expensive per Monte Carlo replicate than two-state's lookup-table policy, so sample sizes below are smaller than two-state's notebook uses; standard errors (SE) are still reported throughout so every point estimate's precision is explicit rather than assumed.

### Convergence of the perturbed objective: $|J^\lambda(\hat\theta_{0.2})-J(\hat\theta_{0.2})|=O(\lambda)$

Theorem "Convergence of the perturbed objective": $|J^\lambda(\theta)-J(\theta)|\le C_T\lambda$ for every $\theta$, with $C_T$ independent of $\lambda$. `n_samples=200,000` (cheap: this is a single batched Monte Carlo evaluation, not a Python-level loop).

In [ ]:
gaps_ref = {lam: objective_gap(env, env.policy_probs, theta_02, mu0_val, DEFAULT_T, lam=lam, sigma=cfg.sigma, n_samples=200_000) for lam in cfg.lambdas}

rows = []
for lam in sorted(gaps_ref):
    g = gaps_ref[lam]
    gap, se = g["gap"].item(), g["J_lambda_se"].item()
    rows.append({"λ": lam, "J(theta_02)": g["J"].item(), "J^lambda(theta_02) (MC)": g["J_lambda_mean"].item(), "SE": se, "|gap|": abs(gap), "|gap|/SE": abs(gap) / se, "|gap|/lambda": abs(gap) / lam})
pd.DataFrame(rows).set_index("λ")

### Gradient-level convergence: $\|\nabla J^\lambda(\hat\theta_{0.2})-\nabla J(\hat\theta_{0.2})\|=O(\lambda)$

Theorem "Gradient-level convergence" (needs the additional smoothness of Assumption "Smoothness of the averaged dynamics": $R_t^\theta$, $K_t^\theta$ continuously differentiable in $\mu$ -- satisfied here, since $P(1|x,a,\mu)=\min\{\mu(1)+\kappa_\mathrm{ad}a,1\}$ is smooth except at the single measure-zero kink $\mu(1)=1-\kappa_\mathrm{ad}a$). The *oracle-D* plug-in estimator (`simplex.gradient_estimate` fed the exact sensitivity flow `exact_sensitivity_flow` instead of the auxiliary plug-in estimate) satisfies $\mathbb E[\hat g^{\mathrm{orc}}_{B,\lambda}(\theta)]=\nabla_\theta J^\lambda(\theta)$ exactly, isolating the perturbation-bias term from sensitivity-estimation noise. `B=cfg.B`, `reps=60`.

In [ ]:
reps_grad = 60
exact_grad_ref = exact_gradient(env, env.policy_probs, theta_02, mu0_val, DEFAULT_T)
oracle_samples_ref, oracle_mean_ref = {}, {}
for lam in cfg.lambdas:
    samples = torch.stack([
        oracle_gradient_estimate(env, env.policy_probs, theta_02, mu0_val, DEFAULT_T, lam=lam, sigma=cfg.sigma, B=cfg.B)
        for _ in range(reps_grad)
    ])
    oracle_samples_ref[lam] = samples
    oracle_mean_ref[lam] = samples.mean(dim=0)

rows = []
for lam in sorted(oracle_mean_ref):
    bias_vec = oracle_mean_ref[lam] - exact_grad_ref
    bias = bias_vec.norm().item()
    bias_se = (oracle_samples_ref[lam].std(dim=0) / reps_grad**0.5).norm().item()
    rows.append({"λ": lam, "||grad J - grad J^lambda||": bias, "SE": bias_se, "bias/SE": bias / bias_se, "bias/lambda": bias / lam})
pd.DataFrame(rows).set_index("λ")

### Population-flow sensitivity estimator: $\hat D_t(k)\to D_t^\theta(k)$

`simplex.estimate_sensitivity_flow`'s single-batch forward estimator $\hat D_t(k)$ of $D_t^\theta(k)=\nabla_\theta\mu_t^\theta(k)$, against the exact value (`exact_sensitivity_flow`, autograd). Bias $A_\eta$ is a property of $\eta$ alone (the single-batch estimator is unbiased for $D_t^{\eta,\theta}$ at *any* $n$, by the conditional-centering Remark); variance $V_\eta/n$ is a property of $n$ alone.

In [ ]:
reps_sens = 150
sens_by_eta = {eta: sensitivity_estimation_error(env, env.policy_probs, theta_02, mu0_val, DEFAULT_T, eta=eta, n=50, sigma=cfg.sigma, reps=reps_sens) for eta in cfg.lambdas}
rows = [{"eta": eta, "n": 50, "bias_norm": r["bias_norm"].sum().item(), "bias_se": r["bias_se"].sum().item(), "resolved (bias>2*SE)": bool(r["bias_norm"].sum().item() > 2 * r["bias_se"].sum().item())} for eta, r in sorted(sens_by_eta.items())]
print(f"bias vs. eta, at n=50 (large relative to cfg.n_aux={cfg.n_aux}, so V_eta/n is small and A_eta is precisely resolved):")
display(pd.DataFrame(rows).set_index("eta"))

n_values = sorted({cfg.n_aux, 5 * cfg.n_aux, 20 * cfg.n_aux})
sens_by_n = {n: sensitivity_estimation_error(env, env.policy_probs, theta_02, mu0_val, DEFAULT_T, eta=0.2, n=n, sigma=cfg.sigma, reps=reps_sens) for n in n_values}
rows = [{"n": n, "eta": 0.2, "bias_norm": r["bias_norm"].sum().item(), "bias_se": r["bias_se"].sum().item(), "variance": r["variance"].sum().item(), "mse": r["mse"].sum().item()} for n, r in sorted(sens_by_n.items())]
print(f"\nvariance vs. n, at fixed eta=0.2 (cfg.n_aux={cfg.n_aux} is the actual training value):")
display(pd.DataFrame(rows).set_index("n"))

### Gradient-estimator bias decomposition: (I) Monte Carlo + (II) sensitivity-estimation + (III) perturbation

Proposition "Mean of the main-batch estimator" decomposes $\hat g_{B,n,\lambda,\eta}(\theta)-\nabla_\theta J(\theta)$ into (I) zero-mean main-batch Monte Carlo noise, (II) the bias from plugging in $\hat D_t$ instead of the exact $D_t^\theta$, and (III) the perturbation bias $\nabla J^\lambda-\nabla J$ above. The ordinary plug-in samples (`gradient_diagnostics`, `n_aux=cfg.n_aux`, `B=cfg.B`, `reps=60`) mix (II) and (III); the oracle-D estimator above isolates (III); their difference approximates (II).

In [ ]:
plugin_ref = {lam: gradient_diagnostics(env, env.policy_probs, theta_02, mu0_val, DEFAULT_T, lam=lam, n_aux=cfg.n_aux, B=cfg.B, sigma=cfg.sigma, reps=reps_grad) for lam in cfg.lambdas}

rows = []
for lam in sorted(oracle_mean_ref):
    term3 = (oracle_mean_ref[lam] - exact_grad_ref).norm().item()
    term23 = plugin_ref[lam]["bias"].norm().item()
    term2_vec = plugin_ref[lam]["mean_estimate"] - oracle_mean_ref[lam]
    term2 = term2_vec.norm().item()
    term2_se = (((plugin_ref[lam]["std"] ** 2 + oracle_samples_ref[lam].std(dim=0) ** 2) / reps_grad).sum() ** 0.5).item()
    rows.append({"λ": lam, "perturbation bias (III)": term3, "(II) approx": term2, "(II) SE": term2_se, "(II)/SE": term2 / term2_se, "total plug-in bias (II)+(III)": term23})
pd.DataFrame(rows).set_index("λ")

### Stability of the perturbed state marginal: $d_{TV}(\nu_t^{\lambda,\theta},\mu_t^\theta)$

Lemma "Stability of the state marginal": $d_{TV}(\nu_t^{\lambda,\theta},\mu_t^\theta)\le L_K\lambda t$, where $\nu_t^{\lambda,\theta}:=\mathrm{Law}(X_t^{\lambda,\theta})$ is the law of the *perturbed* state process (fresh $q_t$ redrawn at every step) and $\mu_t^\theta$ the exact nominal flow. Both the transition kernel and the policy depend on $\mu$ here, so $L_K>0$ genuinely, unlike two-state; growth in $\lambda t$ is expected. `n_samples=200,000` per $(\lambda,t)$.

In [ ]:
rows = []
for lam in sorted(by_lambda):
    tv = state_marginal_stability(env, env.policy_probs, by_lambda[lam]["theta_final"], mu0_val, T=DEFAULT_T, lam=lam, sigma=cfg.sigma, n_samples=200_000)
    rows.append({"λ": lam, **{f"t={t}": tv[t].item() for t in range(tv.shape[0])}})
pd.DataFrame(rows).set_index("λ")

## Summary

Total notebook runtime (including any training performed in this run):

In [ ]:
print(f"total notebook runtime: {time.perf_counter() - _notebook_start:.1f}s")